# Module 3: Commodity News Sentiment Analysis

## Learning Objectives

By completing this notebook, you will:
1. Collect commodity news from multiple sources (NewsAPI, RSS feeds)
2. Use LLMs to analyze sentiment in commodity-specific context
3. Build sentiment scoring systems (bullish/bearish signals)
4. Correlate sentiment with price movements
5. Create real-time sentiment dashboards

## Prerequisites

- Completed Module 2 (RAG Research)
- NewsAPI key (free tier available at https://newsapi.org/)
- Understanding of sentiment analysis concepts
- LLM API access

## Estimated Time: 80-100 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Completed Module 2 (RAG Research)",
    "NewsAPI key (free tier available at https://newsapi.org/)",
    "Understanding of sentiment analysis concepts",
    "LLM API access"
])

## 1. Understanding Commodity Sentiment

**Why Sentiment Matters:**
- News drives short-term price movements
- Sentiment can predict supply disruptions
- Narrative changes affect trader behavior

**Challenges:**
- General sentiment tools miss commodity context
- "Crude oil stockpiles rise" is bearish, but generic sentiment sees "rise" as positive
- Need domain-specific sentiment analysis

**LLM Advantage:**
- Understands commodity market context
- Can explain sentiment reasoning
- Handles complex scenarios

In [ ]:
# Standard library imports
import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

# News APIs
from newsapi import NewsApiClient
import feedparser

# LLM imports
import openai
from anthropic import Anthropic

# Configuration
load_dotenv()
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 7)

# API keys
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
NEWS_API_KEY = os.getenv('NEWS_API_KEY')

# Initialize clients
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
news_client = NewsApiClient(api_key=NEWS_API_KEY) if NEWS_API_KEY else None

# Data directory
DATA_DIR = Path('data/news_sentiment')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")
print(f"   Data directory: {DATA_DIR.absolute()}")
print(f"   News API: {'Configured' if news_client else 'Not configured'}")
print(f"   LLM clients: {'OpenAI' if openai_client else ''} {'Claude' if anthropic_client else ''}")

## 2. Collecting Commodity News

We'll collect news from multiple sources:
- **NewsAPI**: Aggregates from major outlets
- **RSS Feeds**: Direct from Bloomberg, Reuters, etc.
- **Social Media**: Twitter/X commodity traders (optional)

**Search Strategy:**
- Keywords: "crude oil", "WTI", "Brent", "petroleum", "OPEC"
- Time range: Last 7 days for recency
- Language: English
- Sort by: Relevance or publishedAt

In [ ]:
def fetch_commodity_news(query: str, 
                        days_back: int = 7,
                        language: str = 'en') -> List[Dict]:
    """
    Fetch commodity news articles.
    
    Parameters:
    -----------
    query : str
        Search query (e.g., 'crude oil')
    days_back : int
        Number of days to look back
    language : str
        Language code
        
    Returns:
    --------
    List[Dict]
        List of article dictionaries
    """
    if not news_client:
        print("⚠️  NewsAPI not configured")
        return []
    
    # Calculate date range
    to_date = datetime.now()
    from_date = to_date - timedelta(days=days_back)
    
    try:
        # Fetch articles
        response = news_client.get_everything(
            q=query,
            from_param=from_date.strftime('%Y-%m-%d'),
            to=to_date.strftime('%Y-%m-%d'),
            language=language,
            sort_by='relevancy',
            page_size=100
        )
        
        articles = response.get('articles', [])
        
        # Clean and format
        cleaned_articles = []
        for article in articles:
            cleaned_articles.append({
                'title': article.get('title', ''),
                'description': article.get('description', ''),
                'content': article.get('content', ''),
                'url': article.get('url', ''),
                'source': article.get('source', {}).get('name', 'Unknown'),
                'published_at': article.get('publishedAt', ''),
                'author': article.get('author', '')
            })
        
        return cleaned_articles
        
    except Exception as e:
        print(f"❌ Error fetching news: {str(e)}")
        return []

# Fetch crude oil news
print("Fetching crude oil news...\n")
crude_news = fetch_commodity_news('crude oil', days_back=7)

if crude_news:
    print(f"✅ Found {len(crude_news)} articles\n")
    
    # Display sample
    print("Sample articles:")
    for i, article in enumerate(crude_news[:3]):
        print(f"\n[{i+1}] {article['title']}")
        print(f"    Source: {article['source']}")
        print(f"    Date: {article['published_at']}")
        print(f"    Description: {article['description'][:100]}...")
else:
    print("⚠️  No articles fetched. Using sample data for demonstration.")
    
    # Sample articles for demo
    crude_news = [
        {
            'title': 'Oil Prices Rise on OPEC Supply Cuts',
            'description': 'Crude oil futures climbed as OPEC announced extended production cuts.',
            'content': 'OPEC and allies agreed to maintain current production cuts through Q2...',
            'source': 'Reuters',
            'published_at': datetime.now().isoformat(),
            'url': 'https://example.com/1'
        },
        {
            'title': 'US Crude Inventories Decline Sharply',
            'description': 'Weekly EIA data shows larger-than-expected crude stockpile draw.',
            'content': 'US commercial crude oil inventories fell by 6.5 million barrels...',
            'source': 'Bloomberg',
            'published_at': (datetime.now() - timedelta(days=1)).isoformat(),
            'url': 'https://example.com/2'
        },
        {
            'title': 'Hurricane Threatens Gulf Coast Production',
            'description': 'Energy companies evacuate platforms as storm approaches.',
            'content': 'Major oil producers are shutting in production ahead of Hurricane...',
            'source': 'CNBC',
            'published_at': (datetime.now() - timedelta(days=2)).isoformat(),
            'url': 'https://example.com/3'
        }
    ]
    print("\n✅ Using 3 sample articles for demonstration")

## 3. LLM-Based Sentiment Analysis

We'll use LLMs to analyze sentiment with commodity market context.

**Sentiment Scale:**
- Strong Bearish (-2): Major negative developments
- Bearish (-1): Negative indicators
- Neutral (0): Mixed or no clear direction
- Bullish (+1): Positive indicators
- Strong Bullish (+2): Major positive developments

**Key:** LLM provides reasoning, not just a score.

In [ ]:
def analyze_commodity_sentiment(article: Dict) -> Dict[str, Any]:
    """
    Analyze sentiment of commodity news article.
    
    Parameters:
    -----------
    article : dict
        Article with title, description, content
        
    Returns:
    --------
    dict
        Sentiment analysis results
    """
    # Combine article text
    text = f"{article['title']}. {article['description']} {article.get('content', '')}"
    
    prompt = f"""
Analyze the sentiment of this commodity news article from a trader's perspective.

Return JSON:
{{
  "sentiment_score": -2 to +2 (integer: -2=strong bearish, -1=bearish, 0=neutral, +1=bullish, +2=strong bullish),
  "sentiment_label": "strong_bearish|bearish|neutral|bullish|strong_bullish",
  "confidence": 0.0-1.0 (how confident in this assessment),
  "reasoning": "Brief explanation of why this sentiment",
  "price_impact": "likely|possible|minimal",
  "key_factors": ["list of specific factors affecting sentiment"]
}}

Context for commodity traders:
- Supply cuts, production drops, disruptions = BULLISH (higher prices)
- Inventory builds, demand weakness, oversupply = BEARISH (lower prices)
- Consider magnitude and market expectations

Article:
{text[:2000]}
"""
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "You are a commodity market analyst."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            result = json.loads(response.choices[0].message.content)
        
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = message.content[0].text
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0]
            result = json.loads(response_text.strip())
        
        else:
            result = {"error": "No LLM configured"}
        
        return result
        
    except Exception as e:
        return {
            "error": str(e),
            "sentiment_score": 0,
            "sentiment_label": "error"
        }

# Analyze sample articles
print("Analyzing sentiment of crude oil news:\n")
print("="*70)

analyzed_articles = []

for i, article in enumerate(crude_news[:3]):
    print(f"\n[Article {i+1}] {article['title']}")
    print("-"*70)
    
    sentiment = analyze_commodity_sentiment(article)
    
    if 'error' not in sentiment:
        # Add sentiment to article
        article['sentiment'] = sentiment
        analyzed_articles.append(article)
        
        # Display results
        score = sentiment['sentiment_score']
        label = sentiment['sentiment_label'].upper()
        confidence = sentiment.get('confidence', 0)
        
        # Color code
        if score >= 1:
            icon = "📈 🟢"
        elif score <= -1:
            icon = "📉 🔴"
        else:
            icon = "➡️  ⚪"
        
        print(f"{icon} Sentiment: {label} (Score: {score:+d})")
        print(f"   Confidence: {confidence:.1%}")
        print(f"   Price Impact: {sentiment.get('price_impact', 'unknown').upper()}")
        print(f"   Reasoning: {sentiment.get('reasoning', 'N/A')}")
        
        if 'key_factors' in sentiment:
            print(f"   Key Factors:")
            for factor in sentiment['key_factors']:
                print(f"     • {factor}")
    else:
        print(f"❌ Error: {sentiment['error']}")

print("\n" + "="*70)
print(f"Analyzed {len(analyzed_articles)} articles")

### 💡 Exercise 3.1: Aggregate Sentiment Score

**Task:** Calculate an overall market sentiment score from multiple articles.

**Requirements:**
1. Weight each article's sentiment by:
   - Confidence score
   - Recency (more recent = higher weight)
   - Source credibility (optional)
2. Calculate weighted average sentiment
3. Determine overall market signal (bullish/bearish/neutral)
4. Visualize sentiment distribution

**Formula suggestion:**
```
weight = confidence * recency_factor
aggregate_sentiment = sum(score * weight) / sum(weights)
```

**Expected Output:** Aggregate score and distribution chart

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if analyzed_articles:
    print("Calculating Aggregate Sentiment:\n")
    print("="*70)
    
    # Calculate weights
    weighted_scores = []
    weights = []
    
    now = datetime.now()
    
    for article in analyzed_articles:
        sentiment = article['sentiment']
        score = sentiment['sentiment_score']
        confidence = sentiment.get('confidence', 0.5)
        
        # Calculate recency factor (exponential decay)
        pub_date = datetime.fromisoformat(article['published_at'].replace('Z', '+00:00'))
        hours_old = (now - pub_date).total_seconds() / 3600
        recency_factor = np.exp(-hours_old / 48)  # Half-life of 48 hours
        
        # Combined weight
        weight = confidence * recency_factor
        
        weighted_scores.append(score * weight)
        weights.append(weight)
        
        print(f"Article: {article['title'][:50]}...")
        print(f"  Score: {score:+d}, Confidence: {confidence:.2f}, Recency: {recency_factor:.2f}")
        print(f"  Weight: {weight:.3f}\n")
    
    # Calculate aggregate
    aggregate_sentiment = sum(weighted_scores) / sum(weights) if weights else 0
    
    print("="*70)
    print(f"AGGREGATE SENTIMENT: {aggregate_sentiment:+.2f}")
    
    # Determine signal
    if aggregate_sentiment >= 0.5:
        signal = "BULLISH 📈"
    elif aggregate_sentiment <= -0.5:
        signal = "BEARISH 📉"
    else:
        signal = "NEUTRAL ➡️"
    
    print(f"Market Signal: {signal}")
    print("="*70)
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Sentiment distribution
    scores = [a['sentiment']['sentiment_score'] for a in analyzed_articles]
    labels = [a['title'][:30] + '...' for a in analyzed_articles]
    colors = ['red' if s < 0 else 'green' if s > 0 else 'gray' for s in scores]
    
    ax1.barh(labels, scores, color=colors, alpha=0.7)
    ax1.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    ax1.axvline(x=aggregate_sentiment, color='blue', linestyle='-', linewidth=2, label=f'Aggregate: {aggregate_sentiment:+.2f}')
    ax1.set_xlabel('Sentiment Score', fontsize=12)
    ax1.set_title('Individual Article Sentiment', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Sentiment over time
    dates = [datetime.fromisoformat(a['published_at'].replace('Z', '+00:00')) for a in analyzed_articles]
    ax2.scatter(dates, scores, s=200, c=colors, alpha=0.7, edgecolors='black')
    ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax2.axhline(y=aggregate_sentiment, color='blue', linestyle='-', linewidth=2, label=f'Aggregate: {aggregate_sentiment:+.2f}')
    ax2.set_ylabel('Sentiment Score', fontsize=12)
    ax2.set_xlabel('Publication Time', fontsize=12)
    ax2.set_title('Sentiment Over Time', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No analyzed articles available")

## 4. Building a Sentiment Time Series

To track sentiment trends, we need to:
1. Collect news regularly (daily or hourly)
2. Calculate rolling sentiment scores
3. Store in time series database
4. Compare against price movements

**Use Cases:**
- Sentiment momentum (is it getting more bullish?)
- Sentiment divergence (sentiment vs. price)
- Event detection (sudden sentiment shifts)

In [ ]:
def build_sentiment_timeseries(articles: List[Dict], 
                              freq: str = 'D') -> pd.DataFrame:
    """
    Build time series of sentiment scores.
    
    Parameters:
    -----------
    articles : List[Dict]
        Articles with sentiment analysis
    freq : str
        Frequency for resampling ('D'=daily, 'H'=hourly)
        
    Returns:
    --------
    pd.DataFrame
        Time series with sentiment metrics
    """
    if not articles:
        return pd.DataFrame()
    
    # Convert to DataFrame
    records = []
    for article in articles:
        if 'sentiment' in article and 'error' not in article['sentiment']:
            records.append({
                'timestamp': pd.to_datetime(article['published_at']),
                'sentiment_score': article['sentiment']['sentiment_score'],
                'confidence': article['sentiment'].get('confidence', 0.5),
                'title': article['title']
            })
    
    df = pd.DataFrame(records)
    
    if df.empty:
        return df
    
    # Set index
    df = df.set_index('timestamp').sort_index()
    
    # Resample to desired frequency
    resampled = df.resample(freq).agg({
        'sentiment_score': ['mean', 'std', 'count'],
        'confidence': 'mean'
    })
    
    # Flatten column names
    resampled.columns = ['_'.join(col).strip() for col in resampled.columns.values]
    
    # Calculate rolling metrics
    resampled['sentiment_ma3'] = resampled['sentiment_score_mean'].rolling(window=3).mean()
    resampled['sentiment_momentum'] = resampled['sentiment_score_mean'].diff()
    
    return resampled

# Build time series from analyzed articles
if analyzed_articles:
    sentiment_ts = build_sentiment_timeseries(analyzed_articles, freq='D')
    
    print("Sentiment Time Series:")
    print(sentiment_ts)
    
    # Visualize
    if not sentiment_ts.empty and len(sentiment_ts) > 1:
        fig, ax = plt.subplots(figsize=(14, 6))
        
        ax.plot(sentiment_ts.index, sentiment_ts['sentiment_score_mean'], 
                marker='o', linewidth=2, markersize=8, label='Daily Avg Sentiment')
        
        if 'sentiment_ma3' in sentiment_ts.columns:
            ax.plot(sentiment_ts.index, sentiment_ts['sentiment_ma3'], 
                    linestyle='--', linewidth=2, alpha=0.7, label='3-Day MA')
        
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax.fill_between(sentiment_ts.index, 0, sentiment_ts['sentiment_score_mean'],
                        where=(sentiment_ts['sentiment_score_mean'] > 0), alpha=0.3, color='green', label='Bullish')
        ax.fill_between(sentiment_ts.index, 0, sentiment_ts['sentiment_score_mean'],
                        where=(sentiment_ts['sentiment_score_mean'] <= 0), alpha=0.3, color='red', label='Bearish')
        
        ax.set_title('Crude Oil News Sentiment Over Time', fontsize=14, fontweight='bold')
        ax.set_xlabel('Date', fontsize=12)
        ax.set_ylabel('Sentiment Score', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("⚠️  No articles to build time series")

### 💡 Exercise 4.1: Sentiment Dashboard

**Task:** Create a comprehensive sentiment dashboard with multiple metrics.

**Requirements:**
1. Current aggregate sentiment (number and gauge)
2. Sentiment trend (last 7 days)
3. Most bullish and bearish articles
4. Source breakdown (which sources are most bullish/bearish)
5. Key factors mentioned (from sentiment analysis)

**Output:** Professional dashboard display with:
- Summary metrics
- Trend chart
- Top articles table
- Word cloud or factor frequency (optional)

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if analyzed_articles:
    print("\n" + "="*70)
    print("CRUDE OIL SENTIMENT DASHBOARD")
    print("="*70)
    print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Articles Analyzed: {len(analyzed_articles)}")
    print("="*70)
    
    # 1. Current Sentiment
    scores = [a['sentiment']['sentiment_score'] for a in analyzed_articles]
    avg_sentiment = np.mean(scores)
    
    print("\n[1] CURRENT SENTIMENT")
    print("-"*70)
    print(f"   Average Score: {avg_sentiment:+.2f}")
    
    if avg_sentiment >= 0.5:
        print("   Signal: BULLISH 📈🟢")
    elif avg_sentiment <= -0.5:
        print("   Signal: BEARISH 📉🔴")
    else:
        print("   Signal: NEUTRAL ➡️⚪")
    
    bullish = sum(1 for s in scores if s > 0)
    bearish = sum(1 for s in scores if s < 0)
    neutral = sum(1 for s in scores if s == 0)
    
    print(f"   Distribution: {bullish} Bullish, {neutral} Neutral, {bearish} Bearish")
    
    # 2. Top Articles
    print("\n[2] TOP ARTICLES")
    print("-"*70)
    
    sorted_articles = sorted(analyzed_articles, 
                            key=lambda x: abs(x['sentiment']['sentiment_score']), 
                            reverse=True)
    
    print("\n   Most Impactful:")
    for i, article in enumerate(sorted_articles[:3], 1):
        score = article['sentiment']['sentiment_score']
        icon = "🟢" if score > 0 else "🔴" if score < 0 else "⚪"
        print(f"   {i}. {icon} [{score:+d}] {article['title']}")
        print(f"      {article['source']} - {article['sentiment']['sentiment_label']}")
    
    # 3. Key Factors
    print("\n[3] KEY FACTORS MENTIONED")
    print("-"*70)
    
    all_factors = []
    for article in analyzed_articles:
        if 'key_factors' in article['sentiment']:
            all_factors.extend(article['sentiment']['key_factors'])
    
    if all_factors:
        from collections import Counter
        factor_counts = Counter(all_factors)
        print("   Top Factors:")
        for factor, count in factor_counts.most_common(5):
            print(f"     • {factor} (mentioned {count}x)")
    else:
        print("   No factor data available")
    
    # 4. Source Breakdown
    print("\n[4] SENTIMENT BY SOURCE")
    print("-"*70)
    
    source_sentiment = {}
    for article in analyzed_articles:
        source = article['source']
        score = article['sentiment']['sentiment_score']
        if source not in source_sentiment:
            source_sentiment[source] = []
        source_sentiment[source].append(score)
    
    for source, scores in source_sentiment.items():
        avg = np.mean(scores)
        icon = "🟢" if avg > 0 else "🔴" if avg < 0 else "⚪"
        print(f"   {icon} {source}: {avg:+.2f} ({len(scores)} articles)")
    
    print("\n" + "="*70)
    print("End of Dashboard")
    print("="*70)
else:
    print("⚠️  No articles available for dashboard")

## 5. Correlating Sentiment with Price

The ultimate test: does sentiment predict price movements?

**Analysis Approach:**
1. Get historical price data (crude oil futures)
2. Align sentiment scores with price data
3. Calculate correlation
4. Test lead/lag relationships
5. Build predictive features

**Note:** This requires actual historical price data. We'll demonstrate the methodology.

In [ ]:
callout("** This requires actual historical price data. We'll demonstrate the methodology.", "warning")

In [ ]:
# Example: Demonstrate correlation methodology with synthetic price data
if not sentiment_ts.empty:
    print("Sentiment-Price Correlation Analysis\n")
    print("="*70)
    
    # Create synthetic price data for demonstration
    np.random.seed(42)
    price_data = pd.DataFrame({
        'price': 75 + sentiment_ts['sentiment_score_mean'] * 2 + np.random.randn(len(sentiment_ts)) * 1.5,
        'volume': np.random.randint(100000, 500000, len(sentiment_ts))
    }, index=sentiment_ts.index)
    
    # Merge sentiment and price
    combined = sentiment_ts.join(price_data)
    
    print("Combined Data:")
    print(combined[['sentiment_score_mean', 'price']].head())
    
    # Calculate correlation
    correlation = combined['sentiment_score_mean'].corr(combined['price'])
    print(f"\nSentiment-Price Correlation: {correlation:.3f}")
    
    # Test lead/lag
    print("\nLead/Lag Analysis:")
    for lag in [0, 1, 2]:
        if lag == 0:
            corr = correlation
            label = "Same day"
        else:
            lagged_price = combined['price'].shift(-lag)
            corr = combined['sentiment_score_mean'].corr(lagged_price)
            label = f"Sentiment leads by {lag} day(s)"
        
        print(f"  {label}: {corr:.3f}")
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # Sentiment
    ax1.plot(combined.index, combined['sentiment_score_mean'], 
             marker='o', color='blue', linewidth=2, label='Sentiment')
    ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax1.set_ylabel('Sentiment Score', fontsize=12)
    ax1.set_title('Sentiment vs. Price', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Price
    ax2.plot(combined.index, combined['price'], 
             marker='s', color='green', linewidth=2, label='Price (synthetic)')
    ax2.set_ylabel('Price ($)', fontsize=12)
    ax2.set_xlabel('Date', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("Note: This uses synthetic price data for demonstration.")
    print("In production, integrate with real price data APIs (e.g., Alpha Vantage, Yahoo Finance)")
    print("="*70)
else:
    print("⚠️  No sentiment time series available")

## Summary

### Key Takeaways

1. **LLMs Excel at Context-Aware Sentiment** - Understand commodity-specific nuances

2. **Aggregation Requires Weighting** - Consider confidence, recency, and source quality

3. **Time Series Analysis Reveals Trends** - Momentum matters as much as levels

4. **Price Correlation Validates Signals** - Test sentiment against actual market movements

5. **Real-Time Processing Enables Trading** - Automated sentiment pipelines can trigger actions

### Skills Gained

✅ Collecting commodity news from APIs  
✅ LLM-based sentiment analysis with reasoning  
✅ Building aggregate sentiment scores  
✅ Creating sentiment time series  
✅ Correlating sentiment with price movements  
✅ Designing sentiment dashboards  

### What's Next

In subsequent modules, you'll:
- Integrate sentiment into trading strategies
- Build multi-asset sentiment monitors
- Create real-time alert systems
- Deploy production sentiment pipelines

### Additional Resources

- **NewsAPI Documentation:** https://newsapi.org/docs
- **Sentiment Analysis Best Practices:** Domain-specific models
- **Financial NLP:** Understanding commodity-specific language

---

**Continue with remaining modules to build complete commodity trading systems**